[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/https://github.com/maurergroup/ML_in_CP_UVie_SS26/Module_I_Introduction/WS1_Introduction/WS1_Introduction.ipynb)

# WS 1 — Introduction to Machine Learning Basics

**Machine Learning in Computational Physics** · University of Vienna

---

This workshop introduces the key components of a machine learning workflow. We are doing this with a classic image classification example rather than a physics-based example (We will cover those in all subsequent workshops). We will be using the PyTorch framework, but it would not look dramatically different in JAX, scikit-learn, Tensorflow or MLJ.jl (Machine Learning in Julia). In later workshops, we will also use scikit-learn.

### Learning Objectives

By the end of this workshop you will be able to:

- Load and prepare a dataset using `Dataset` and `DataLoader`
- Define a machine learning model (neural network) by subclassing `nn.Module` (We will cover NNs in more detail in Module V)
- Write a training loop with backpropagation
- Track and visualise training progress (loss and accuracy)
- Save and reload model weights from disk
- Understand how to move computation to a GPU

### Building blocks of a ML workflow

- dataset and data handling/pipeline
- ML model
- Loss function
- Optimiser
- training pipeline


## 0. Setup

We load all relevant packages and deploy torch onto a GPU if we have one available. Otherwise, we load it onto a CPU

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import wandb

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import torchvision.transforms as transforms

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f'Using device: {device}')
print(torch.cuda.is_available())
print("Your current machine may have a CUDA device available, but let's stick to a cpu for now")
device="cpu"

## 1. Data — Loading and Exploration

PyTorch provides two key data primitives:

- **`Dataset`** — wraps your data and its labels; knows how to return a single sample via `__getitem__`.
- **`DataLoader`** — wraps a `Dataset` and handles batching, shuffling, and parallel loading.

### FashionMNIST

[FashionMNIST](https://github.com/zalandoresearch/fashion-mnist) contains 70 000 greyscale images (28 × 28 px) of clothing items in 10 categories. `torchvision.datasets` downloads and caches it automatically. This dataset is brought to you by the data scientists of Zalando!

![FashionMNIST](https://github.com/zalandoresearch/fashion-mnist/raw/master/doc/img/fashion-mnist-sprite.png)

| Label | Class | Label | Class |
|-------|-------|-------|-------|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

In [ ]:
# ToTensor() converts PIL images to torch.Tensor with shape (C, H, W), which stand for channels, height, and width, respectively. It also converts the data type to float32 and scales pixel values from [0, 255] to [0.0, 1.0].
training_data = datasets.FashionMNIST(
    root='data', train=True,  download=True, transform=ToTensor()
)
test_data = datasets.FashionMNIST(
    root='data', train=False, download=True, transform=ToTensor()
)
# check out https://docs.pytorch.org/vision/main/generated/torchvision.datasets.FashionMNIST.html for more details on the dataset and its parameters. train=True pulls the training set, while train=False pulls the test set. The dataset will be downloaded to the specified root directory if it is not already present. The transform parameter applies the ToTensor transformation to the images, converting them to PyTorch tensors.

print(f'Training samples : {len(training_data)}')
print(f'Test samples     : {len(test_data)}')
print(f'Image shape      : {training_data[0][0].shape}  (channels, height, width)')
print(f'Number of classes: {len(training_data.classes)}')

We can see that the images have a data shape of 1 channel by 28 by 28 pixels. The label y is a discrete value (np.int).

<div class="alert alert-block alert-info">

**TASK**: Explore the dataset object `training_data` by checking out the functions and objects it contains. 
Type
```
training_data.
```
and use TAB to autocomplete
    
</div>



In [ ]:
training_data[0][0] #access the input X of the 1st datapoint
training_data[0][1] #access the label y of the 1st datapoint

class_names = training_data.classes
print(class_names)

#your code here

Now we use the DataLoader to prepare the dataset for the machine learning workflow. We define the batch size, which is the "portion size" that the model will be trained on during a training step. We do this for the training set and the test set.

In [ ]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader  = DataLoader(test_data,     batch_size=batch_size)

# DataLoaders are iterable, so we can get a batch of data by using the iter() function and next() function. The iter() function returns an iterator object, and the next() function returns the next item from the iterator. In this case, we get a batch of data (X_batch, y_batch) from the train_dataloader.

X_batch, y_batch = next(iter(train_dataloader))
print(f'Batch feature shape : {X_batch.shape}  (batch, channels, height, width)')
print(f'Batch label shape   : {y_batch.shape}')

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4.2))
indices = np.random.choice(len(training_data), 16, replace=False)
for ax, idx in zip(axes.flat, indices):
    image, label = training_data[idx]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(class_names[label], fontsize=8)
    ax.axis('off')
plt.suptitle('Random training samples', y=1.02)
plt.tight_layout()
plt.show()

### Exercise 1 — Explore the data *(own time)*

<div class="alert alert-block alert-info">

**TASK**
1. Count how many samples each class has in the training set. Are the classes balanced?
2. Compute and display the **mean image** for each of the 10 classes (average over all samples in a class).
3. Experiment with data augmentation: apply `transforms.RandomHorizontalFlip()` and `transforms.RandomRotation(30)` to the training data. What effect does this have on the training images?

</div>


**Tip**: Use u:ai to get tips on how to do these tasks or google it. You have to transform the image to a tensor at the end. You can apply multiple transformations by building a transform pipeline

```
# Build a transform pipeline: rotate up to ±30°, then convert to tensor
augment_transform = transforms.Compose([
    transforms.RandomRotation(degrees=30),
    transforms.ToTensor(),
])
```
You then reload the dataset with the modified transform.

In [ ]:
#your code here

#point 1

In [ ]:
#your code here

#point 2


In [ ]:
#your code here

#point 3

## 2. Model — A simple Neural Network

Neural networks in PyTorch are Python classes that subclass `nn.Module`. We are using a simple neural network as an example of a machine learning model without dwelling on the details. You will learn more about them in Module V. 

We implement the neural network as a class with two functions:

- `__init__` — define all layers and sub-modules as attributes.
- `forward` — defines how input data flows through the network (forward pass only) to give the output labels.

PyTorch uses automatic differentiation. It's **autograd** engine records operations during the forward pass and uses them to compute gradients automatically during backpropagation — you never write the backward pass yourself. Automatic differentiation and the backpropagation is key to update model parameters during a training step.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()          # transforms 28x28 -> to vector with 784 features
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),              # 10 output logits (one per class)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

This is the core of the model, where we define a sequence of layers. The first layer is a linear layer that takes the 784 input features and maps them to 512 hidden units. This is followed by a ReLU activation function, which introduces non-linearity. The second linear layer maps the 512 hidden units to another set of 512 hidden units, followed by another ReLU activation. Finally, the last linear layer maps the 512 hidden units to 10 output `logits`, one for each class in the FashionMNIST dataset. The outputs can have random positive or negative values. These can be mapped onto a (0,1) domain to describe the probability of the input image belonging to one of the classes.  

In [ ]:
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total:,}')
print(f'Trainable parameters : {trainable:,}')


The trainable parameters arise from the model definition and are calculated based on the number of weights and biases in each layer. 
Each linear layer looks like this:

$$ y = W\cdot x + B $$

where W is a matrix of weights and B is a vector of biases.

For example, the first linear layer has 28x28x512 weights and 512 biases, the second linear layer has 512x512 weights and 512 biases, and the final linear layer has 512x10 weights and 10 biases. The total number of parameters is the sum of all these weights and biases across all layers.

The ReLU has no trainable parameters. It just applies a non-linear operation on the input:

$$ \text{ReLU} = \text{max}(0,x)  $$

In [ ]:
# Let's trace a single random image through the network step by step.
x = torch.rand(1, 28, 28, device=device)
print(f'Input shape              : {x.shape}')

x_flat = nn.Flatten()(x)
print(f'After Flatten            : {x_flat.shape}')

x_h1 = nn.Linear(784, 512).to(device)(x_flat)
print(f'After Linear(784 -> 512) : {x_h1.shape}')

x_h1_act = nn.ReLU()(x_h1)
print(f'After ReLU               : {x_h1_act.shape}')
print(f'  min before ReLU: {x_h1.min().item():.3f}   min after: {x_h1_act.min().item():.3f}  (negatives clipped)')

In [ ]:
sample = torch.rand(1, 28, 28, device=device)
logits = model(sample)
probs  = nn.Softmax(dim=1)(logits) # The Softmax function converts the raw output logits into probabilities that sum to 1 across the classes. The dim=1 argument specifies that the softmax should be applied across the class dimension (the second dimension of the output), which is necessary for multi-class classification tasks.

print(f'Raw logits    : {logits.detach().cpu().numpy().round(3)}')
print(f'Probabilities : {probs.detach().cpu().numpy().round(3)}')
print(f'Sum of probs  : {probs.sum().item():.4f}')
print(f'Predicted     : {class_names[probs.argmax(1).item()]}')

## 3. Defining the Loss function and Optimiser




In [ ]:
learning_rate = 1e-3

loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

print(f'Loss function : {loss_fn}')
print(f'Optimizer     : {optimizer}')


**`CrossEntropyLoss`** nn.CrossEntropyLoss combines nn.LogSoftmax and nn.NLLLoss (negative log-likelihood) in one numerically stable operation. It expects raw **logits** as input — so our model correctly returns logits from its final layer rather than probabilities.

The negative log-likelihood is defined as $-\text{log}(p)$, where p is the probability outputted by the softmax.

**`SGD`** (Stochastic Gradient Descent) is one of the many optimisers that pyTorch sypports (https://docs.pytorch.org/docs/stable/optim.html#algorithms). It updates each parameter in the direction that reduces the loss:

$$\theta \leftarrow \theta - \alpha \, \nabla_\theta \mathcal{L}$$

where $\alpha$ is the learning rate.

## 4. Defining the training and evaluation workflows

Training a neural network iterates three steps per batch:

1. **Forward pass** — compute predictions and measure the loss.
2. **Backward pass** — compute gradients with respect to all parameters (`loss.backward()`).
3. **Parameter update** — Use gradients to adjust weights in the direction that reduces the loss (`optimizer.step()`).

Now we combine all ingredients to the function that performs a single training epoch. In each epoch, we go through all batches in the dataloader. For each batch, we do the model prediction, evaluate the loss, backpropagate to get the gradients and apply the optimisation step. `optimizer.zero_grad()` clears accumulated gradients before each batch because PyTorch accumulates them by default.

During evaluation (either on the train or test data), we go through all the data in the loader, do the prediction, calculate the loss (if we have ground truth labels) and evaluate the accuracy of the prediction. There are many different accuracy measures (mean absolute error, root mean squared error, etc.). In this example, we calculate the accuracy with the **top-1** accuracy measure. It only considers the prediction quality of the highest probability logit.

$$\text{top-1 accuracy} = \frac{\text{number of exactly correct predictions with highest probability logit}}{\text{total samples}}$$



In [ ]:
def train_one_epoch(dataloader, model, loss_fn, optimizer, device):
    """Train for one epoch. Returns mean batch loss."""
    model.train()
    size = len(dataloader.dataset)
    total_loss = 0.0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    return total_loss / len(dataloader)

def evaluate(dataloader, model, loss_fn, device):
    """Evaluate model on a dataloader. Returns (accuracy, mean loss)."""
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
    return correct / len(dataloader.dataset), total_loss / len(dataloader)

Now we train the model for the predefined number of epochs

In [ ]:
learning_rate = 1e-3
epochs        = 5

# Re-initialise model and optimizer for a clean run.
model     = NeuralNetwork().to(device) #without this, running this cell multiple times will continue training the same model.
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

train_losses, test_losses, test_accs = [], [], []

print(f'Training on {device} for {epochs} epochs')
print('-' * 55)
for epoch in range(1, epochs + 1):
    train_loss          = train_one_epoch(train_dataloader, model, loss_fn, optimizer, device) #train one epoch and calcualte train loss
    test_acc, test_loss = evaluate(test_dataloader, model, loss_fn, device) #calculate test accuracy and test loss after each epoch
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    print(f'  Epoch {epoch:2d}  train_loss={train_loss:.4f}  test_loss={test_loss:.4f}  accuracy={100*test_acc:.1f}%')

print()
print(f'Final test accuracy: {100 * test_accs[-1]:.1f}%')

### Exercise — Hyperparameter search *(own time)*

<div class="alert alert-block alert-info">

**TASK**

1. Double and halve the learning rate. How does it affect training stability and final accuracy?
2. Increase `epochs` to 20.
3. Try another optimisation algorithm implemented in pyTorch, e.g. `torch.optim.Adam(model.parameters(), lr=1e-3)`. Does it converge faster?
4. Play with hyperparameters to get the best possible `test_loss`.  Visualize the training as a function of epoch with the plotting functions below.
</div>




In [ ]:
#your code here

## 5. Visualisation — Inspecting Training Progress

In [ ]:
ep = range(1, epochs + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ep, train_losses, label='train')
ax1.plot(ep, test_losses,  label='test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss over training')
ax1.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax1.legend()

ax2.plot(ep, [100 * a for a in test_accs], color='tab:green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Test accuracy')
ax2.set_ylim(0, 100)
ax2.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
imgs, true_labels = next(iter(test_dataloader))
imgs        = imgs[:16].to(device)
true_labels = true_labels[:16].to(device)

with torch.no_grad():
    pred_labels = model(imgs).argmax(1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4.2))
for ax, img, true, pred in zip(axes.flat, imgs.cpu(), true_labels.cpu(), pred_labels.cpu()):
    ax.imshow(img.squeeze(), cmap='gray')
    colour = 'green' if true == pred else 'red'
    ax.set_title(class_names[pred.item()], fontsize=8, color=colour)
    ax.axis('off')
plt.suptitle('Predictions — green = correct, red = wrong', y=1.02)
plt.tight_layout()
plt.show()

### Weights & Biases – Interactive training dashboard

Matplotlib plots are static snapshots after training has finished. For more complicated models that take longer to train, it would be nice to see how this is progressing before it's finished. 
[**Weights & Biases**](https://www.wandb.ai) is an interactive online dashboard that can update in real time as training runs. It can also be used to manage large numbers of training runs, optimise model parameters, and as a repository for finished models, so important results don't get lost. 

**Try setting up an account on the website, then run the cells below, following the instructions to log in with your account.**

A number of different tools have been developed for this sort of thing, including [TensorBoard](https://www.tensorflow.org/tensorboard/get_started) and [MLFlow](https://mlflow.org). 

In [ ]:
import wandb

wandb.login()

In [ ]:
%%wandb
# Re-train from scratch, logging to Weights & Biases each epoch
tb_model     = NeuralNetwork().to(device)
tb_optimizer = torch.optim.SGD(tb_model.parameters(), lr=learning_rate)
tb_epochs    = 10 # Increase this so you can see it running live. 

wb_run = wandb.init(
    project="ml-in-cp",
    # name="ws1-intro-example", # Leave this out to automatically generate a name. We set the name on purpose to show results from it later. 
    config={ # Add hyperparameters you're interested in tracking here. These will be visible in W&B and can be used to filter and compare runs.
        "learning_rate": learning_rate,
        "epochs": tb_epochs,
        "batch_size": batch_size,
        "device": device,
        "dataset": "FashionMNIST",
    },
)

for epoch in range(1, tb_epochs + 1):
    train_loss          = train_one_epoch(train_dataloader, tb_model, loss_fn, tb_optimizer, device)
    test_acc, test_loss = evaluate(test_dataloader, tb_model, loss_fn, device)

    # Log information to W&B
    wb_run.log({
        "train_loss": train_loss,
        "test_loss": test_loss,
        "test_accuracy": 100 * test_acc
    })

    # Log weight histograms — visible under the 'Histograms' tab
    for name, param in tb_model.named_parameters():
        wb_run.log({f'weights/{name}': wandb.Histogram(param.cpu().detach().numpy())}, step=epoch)

# Log a batch of test images with their predicted labels — visible under the 'Images' tab
tb_model.eval()
imgs_tb, labels_tb = next(iter(test_dataloader))
imgs_tb = imgs_tb[:16].to(device)

with torch.no_grad():
    preds_tb = tb_model(imgs_tb).argmax(1)

# make_grid arranges a list of tensors into a single grid image
from torchvision.utils import make_grid
grid = make_grid(imgs_tb.cpu(), nrow=8, normalize=True)
wb_run.log({"examples": wandb.Image(grid, caption=[f'True: {class_names[true]} | Pred: {class_names[pred]}' for true, pred in zip(labels_tb[:16], preds_tb.cpu()[:16])])})

# Save model as an artifact - this lets you download restore specific versions of the model later on. 
torch.save(tb_model.state_dict(), 'fashionmnist_model.pth')
model_artifact = wandb.Artifact('fashionmnist-cnn', type='model')
model_artifact.add_file('fashionmnist_model.pth') 
wb_run.log_artifact(model_artifact)
wb_run.finish() # Finish the W&B run to ensure all data is uploaded and the run is marked as complete.

## 6. Persistence — Saving and Loading Models

The recommended approach is to save only the model's **`state_dict`** — the dictionary of weight tensors — rather than the whole model object. This keeps the saved file stable even if the class definition changes later.

When reloading, always call `model.eval()` to switch off training-time behaviour and automatic differentiation before running inference. This speeds up the model considerably.

In [ ]:
torch.save(model.state_dict(), 'fashionmnist_model.pth')
print('Saved model weights to fashionmnist_model.pth')

In [ ]:
loaded_model = NeuralNetwork().to(device) #reinitalise the model architecture before loading the weights
loaded_model.load_state_dict(torch.load('fashionmnist_model.pth', weights_only=True)) #load the weights into the model
loaded_model.eval()   # switch off training-time behaviour (dropout, batchnorm, autograd, etc.)
print('Model loaded and ready for inference.')

In [ ]:
pred_orig   = model(imgs).argmax(1)
pred_loaded = loaded_model(imgs).argmax(1)

match = (pred_orig == pred_loaded).all().item()
print(f'Original and loaded model give identical predictions: {match}')

## 7. Hardware — CPU vs GPU

The same PyTorch code runs unchanged on CPU and GPU. The only difference is the **device string** used to move tensors and models:

| Operation | Code |
|-----------|------|
| Move model to GPU | `model.to("cuda")` |
| Move tensor to GPU | `tensor.to("cuda")` |
| Move result back to CPU | `tensor.cpu()` |

Best practice: define `device` once at the top of the notebook (as we did in Section 0) and pass it through everywhere — never hard-code `"cpu"` or `"cuda"` in multiple places.

**For Mac users:** Use `"mps"` instead of `"cuda"` to make use of your Apple Silicon GPU. However, keep in mind that this is an experimental feature and not all code will work. 

In [ ]:
n_timing_epochs = 3

def timed_training_cpu(n_epochs):
    m   = NeuralNetwork().to('cpu')
    opt = torch.optim.SGD(m.parameters(), lr=1e-3)
    t0  = time.perf_counter()
    for _ in range(n_epochs):
        train_one_epoch(train_dataloader, m, loss_fn, opt, 'cpu')
    return time.perf_counter() - t0

def timed_training_gpu(n_epochs):
    m   = NeuralNetwork().to('cuda')
    opt = torch.optim.SGD(m.parameters(), lr=1e-3)
    # torch.cuda.Event uses CUDA's own hardware timer — the only reliable
    # way to time GPU work, since CUDA kernels launch asynchronously.
    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(n_epochs):
        train_one_epoch(train_dataloader, m, loss_fn, opt, 'cuda')
    end.record()
    torch.cuda.synchronize()  # block until end event is reached
    return start.elapsed_time(end) / 1000  # ms -> s

cpu_time = timed_training_cpu(n_timing_epochs)
print(f'CPU  -  {n_timing_epochs} epochs: {cpu_time:.2f} s')

if torch.cuda.is_available():
    timed_training_gpu(1)  # warm-up: initialise CUDA kernels before measuring
    gpu_time = timed_training_gpu(n_timing_epochs)
    print(f'GPU  -  {n_timing_epochs} epochs: {gpu_time:.2f} s')
    print(f'Speedup: {cpu_time / gpu_time:.2f}x')
    print()
    print('Note: speedup is modest for small MLPs — the network is too small to')
    print('saturate GPU cores and CPU→GPU data transfer adds overhead.')
else:
    print('No CUDA GPU detected on this machine.')

> **Note:** For a small fully-connected network like this one the GPU advantage is modest — the model is too small to saturate thousands of GPU cores. The benefit becomes dramatic for larger models (CNNs, Transformers) and bigger batches.